# DataMate ZAIC Hybrid Pipeline (Kaggle Edition)

Quy trình Text-to-SQL Hybrid: Ingestion DuckDB, SQLSolver, Semantic Views, Debt Layer và Self-Consistency Voting.

**Chuẩn bị:** Upload thư mục datamate_zaic_2026/ thành một Kaggle Dataset và đính kèm vào notebook.
- Backend Gemini: Internet = On + Add Secret GEMINI_API_KEY (không cần GPU).
- Backend Local (Qwen3.5-9B): Accelerator = GPU (T4 x2 hoặc P100 / A100) + Internet = On.

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

# Tìm thư mục chứa package datamate trong Kaggle inputs hoặc working dir
candidates = (
    glob.glob('/kaggle/input/**/datamate/__init__.py', recursive=True)
    + glob.glob('/kaggle/input/**/app/__init__.py', recursive=True)
    + [str(Path.cwd() / 'datamate/__init__.py'), str(Path.cwd().parent / 'datamate/__init__.py')]
)

found = [c for c in candidates if os.path.exists(c)]
if found:
    ROOT = found[0].replace('\\', '/').rsplit('/datamate/', 1)[0].rsplit('/app/', 1)[0]
else:
    ROOT = '/kaggle/working/datamate_zaic_2026'

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass
print(f'Working Directory & ROOT: {ROOT}')

BACKEND = 'gemini'          # 'gemini' | 'local'

# Cài đặt dependencies cần thiết cho môi trường Kaggle
req_packages = ['duckdb>=1.4.0,<=2.0.0', 'pyarrow', 'xlrd>=2.0.1', 'openpyxl>=3.1.0']
if BACKEND == 'gemini':
    req_packages.append('google-genai')
else:
    req_packages.extend(['transformers>=4.48', 'accelerate', 'peft', 'bitsandbytes'])

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', *req_packages], check=False)

In [ ]:
from datamate import Config
from datamate.agent.pipeline import run_inference
from datamate.llm import create_backend
from evaluation.run_eval import run_train

if BACKEND == 'gemini':
    cfg = Config(backend='gemini', max_biz=2, per_biz=15, limit_test=20)      # ~30 câu train + 20 câu test
else:
    cfg = Config(backend='local', max_biz=1, per_biz=10, limit_test=20)       # đo thời gian một câu trước đã
backend = create_backend(cfg)

## Chạy + chấm PA trên một phần train

In [ ]:
solver, split, ev, subs, logs = run_train(cfg, backend)

Các câu sai (mỗi câu −1) kèm code đã sinh:

In [ ]:
q_text = split.questions.set_index("id")["question"]
by_id = {lg["id"]: lg for lg in logs}
for r in ev[ev.status == "wrong"].head(15).itertuples():
    print(f"[{r.id}] {q_text[r.id]}\n   pred=({r.type}) {r.answer!r} | gold=({r.gold_type}) {r.gold!r} | agree={r.agree}")
    if by_id[r.id]["codes"]: print("   code:", by_id[r.id]["codes"][0][:600].replace("\n", "\n         "))


## (tùy chọn) Xuất dữ liệu SFT từ lời giải đã đúng trên train

In [ ]:
from datamate.agent.context import ContextBuilder
from training.export_sft import export_sft
n = export_sft(ev, logs, split, ContextBuilder(cfg, backend.count_tokens), f'{cfg.out_dir}/sft_train.jsonl')
print('Đã ghi', n, 'mẫu SFT')

## Dự đoán public test và xuất `submission.jsonl` (đủ mọi id, có kiểm định)

In [ ]:
subs_te, logs_te, problems = run_inference(cfg, backend)